In [1]:
# ============================================================
# CELL 28 — Reserved core comparative evaluation
# ============================================================
# Reference: EVALUATION_PROTOCOL.md, Sections 1–3.
#
# Reuse the frozen development implementation.
# Preserve the preceding 65-batch commitment history.
# Evaluate reserved batches 65–129.
# Do not adjust scenarios, reference outcomes or verifier rules.

import json
import hashlib
import io
from pathlib import Path
from contextlib import redirect_stdout


# ------------------------------------------------------------
# 1. Enforce the experimental freeze
# ------------------------------------------------------------

evaluation_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "EVALUATION_PROTOCOL.md").is_file()
)

evaluation_notebook = (
    evaluation_root
    / "notebooks/03_benchmark_evaluation.ipynb"
)

development_notebook = (
    evaluation_root
    / "notebooks/02_benchmark_development.ipynb"
)

freeze_manifest_path = (
    evaluation_root / "EVALUATION_FREEZE.json"
)

assert freeze_manifest_path.is_file(), (
    "Final experimental freeze is not complete. "
    "Do not execute the reserved evaluation."
)


def file_digest(path):
    """Return the SHA-256 fingerprint of a file."""

    return hashlib.sha256(path.read_bytes()).hexdigest()


def notebook_sources(path):
    """Read nonempty code cells in their saved order."""

    document = json.loads(path.read_text(encoding="utf-8"))

    return [
        "".join(cell["source"])
        for cell in document["cells"]
        if cell["cell_type"] == "code"
        and "".join(cell["source"]).strip()
    ]


def source_digest(sources):

    encoded = json.dumps(
        sources,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


evaluation_freeze = json.loads(
    freeze_manifest_path.read_text(encoding="utf-8")
)

assert file_digest(development_notebook) == (
    evaluation_freeze["development_notebook_sha256"]
)

assert file_digest(
    evaluation_root / "EVALUATION_PROTOCOL.md"
) == evaluation_freeze["evaluation_protocol_sha256"]

assert source_digest(
    notebook_sources(evaluation_notebook)
) == evaluation_freeze["evaluation_code_sha256"]

assert len(notebook_sources(evaluation_notebook)) == 4

print("=== EXPERIMENTAL FREEZE ===")
print("Development notebook fingerprint: PASS")
print("Evaluation protocol fingerprint: PASS")
print("Evaluation code fingerprint: PASS")


# ------------------------------------------------------------
# 2. Reuse the complete frozen development implementation
# ------------------------------------------------------------

frozen_sources = notebook_sources(development_notebook)

assert len(frozen_sources) == 27

print("\n=== FROZEN IMPLEMENTATION INITIALIZATION ===")

for index, source in enumerate(frozen_sources, start=1):

    captured = io.StringIO()

    try:

        with redirect_stdout(captured):

            exec(
                compile(
                    source,
                    f"Frozen development cell {index}",
                    "exec",
                ),
                globals(),
            )

    except Exception:

        print(captured.getvalue())
        raise

    print(f"Development cell {index}/27: EXECUTED")


# Verify the reproduced development fingerprints.
assert development_code_digest == (
    "d454f92af595566d62ba570d194226516"
    "fe74afdffebb525d681160693da59d4"
)

assert reference_definition_digest == (
    "e438851e88dc6efc6d150ae8993e6da3b"
    "4f70bb25e6c8c97493f0964be564233"
)

assert development_result_digest == (
    "2398af4d60a5f506dcb373b0a4a51031"
    "ce619bec3f986acba712d4eece5568cf"
)

print("Frozen development fingerprints: EXACT MATCH")


# ------------------------------------------------------------
# 3. Load the reserved partition
# ------------------------------------------------------------

def load_reserved_batches():
    """Reuse the verified CSV boundaries and batch definitions."""

    proofs = json.loads(
        (RAW / "Meta Data/proof_log.json").read_text()
    )

    records_by_file = {}

    for file in sorted((RAW / "logs").glob("*.csv")):

        with file.open(
            encoding="utf-8",
            newline="",
        ) as stream:

            records_by_file[file.name] = list(
                csv.DictReader(stream)
            )

    batches = []

    observed_hashes = set()

    previous_end = max(partition_times["development"])

    for sequence in evaluation_batch_indices:

        proof = proofs[sequence]

        filename = Path(proof["csv_path"]).name

        start = proof["row_start"]
        end = proof["row_end"]

        assert 0 <= start <= end < len(
            records_by_file[filename]
        )

        records = records_by_file[filename][start:end + 1]

        assert len(records) == proof["size"]

        timestamps = [
            datetime.fromisoformat(record["timestamp"])
            for record in records
        ]

        assert all(
            current > previous
            for previous, current in zip(
                timestamps, timestamps[1:]
            )
        )

        assert timestamps[0] > previous_end

        previous_end = timestamps[-1]

        hashes = {
            record["chain_hash"] for record in records
        }

        assert len(hashes) == len(records)

        assert not observed_hashes.intersection(hashes)

        observed_hashes.update(hashes)

        batches.append({
            "batch_index": sequence,
            "proof": proof,
            "records": records,
        })

    assert len(batches) == EVALUATION_COUNT

    assert observed_hashes == evaluation_hashes

    assert observed_hashes.isdisjoint(development_hashes)

    return batches


evaluation_batches = load_reserved_batches()

print("\n=== RESERVED PARTITION ===")
print("Batches:", len(evaluation_batches))
print("First batch:", evaluation_batches[0]["batch_index"])
print("Final batch:", evaluation_batches[-1]["batch_index"])
print("Measurement records:", sum(
    len(batch["records"])
    for batch in evaluation_batches
))
print("Original record identity verification: PASS")


# ------------------------------------------------------------
# 4. Preserve the preceding commitment history
# ------------------------------------------------------------

evaluation_anchor = EthereumAnchor()

evaluation_prefix = list(log_entries)

evaluation_prior_bundle = deepcopy(
    witnessed_checkpoints[-1]
)

original_development_state = anchor_state(
    ethereum_anchor
)

for sequence, commitment in enumerate(log_entries):

    receipt = evaluation_anchor.append(
        sequence,
        commitment,
        evaluation_anchor.head(),
    )

    assert receipt["status"] == 1

assert anchor_state(evaluation_anchor) == (
    original_development_state
)

previous_commitment = (
    signed_development_evidence[-1]["commitment"]
)

previous_record_hash = (
    development_batches[-1]["records"][-1]["chain_hash"]
)

print("\n=== PRECEDING HISTORY ===")
print("Preserved commitments:", evaluation_anchor.count())
print("Preceding application history: PASS")


# ------------------------------------------------------------
# 5. Construct reserved baseline evidence
# ------------------------------------------------------------

evaluation_envelopes = {}

evaluation_certificates = {}

evaluation_contexts = {}

evaluation_histories = {}

evaluation_record_predecessors = {}

for batch in evaluation_batches:

    sequence = batch["batch_index"]

    evaluation_record_predecessors[sequence] = (
        previous_record_hash
    )

    assert rehash_source_records(
        batch["records"],
        previous_record_hash,
    ) == batch["records"]

    evidence = build_evidence(
        batch,
        previous_commitment,
    )

    commitment = evidence_commitment(evidence)

    private_key = source_private_keys[
        evidence["source_id"]
    ]

    envelope = {
        "evidence": evidence,
        "commitment": commitment,
        "public_key": public_key_bytes(private_key),
        "signature": sign_evidence(evidence, private_key),
    }

    context = build_policy_context(batch)

    evaluation_envelopes[sequence] = envelope

    evaluation_contexts[sequence] = context

    evaluation_certificates[sequence] = (
        build_certificate(context)
    )

    evaluation_histories[sequence] = VerifierHistory(
        next_sequence=sequence,
        previous_commitment=previous_commitment,
    )

    previous_commitment = commitment

    previous_record_hash = batch["records"][-1]["chain_hash"]

assert len(evaluation_envelopes) == EVALUATION_COUNT

print("Reserved baseline envelopes:", len(evaluation_envelopes))
print("Original measurement hash-chain verification: PASS")


# ------------------------------------------------------------
# 6. Execute frozen core scenarios
# ------------------------------------------------------------

evaluation_comparative_results = []

evaluation_control_cases = {}

evaluation_checkpoints = {}

evaluation_branch_count = 0

evaluation_rollback_count = 0


def collect_reserved_case(
    scenario_id,
    injected,
    case,
):
    """Collect decisions while the corresponding branch is active."""

    sequence = case["policy_context"].expected_sequence

    observations = collect_comparative_results(
        sequence,
        scenario_id,
        case,
    )

    evaluation_comparative_results.extend(observations)

    if scenario_id == "T00":

        # T00 belongs to the canonical commitment branch.
        # Its restricted view remains valid as that history
        # advances. Never retain views from rolled-back branches.
        assert sequence not in evaluation_control_cases

        evaluation_control_cases[sequence] = dict(case)


print("\n=== RESERVED CORE COMPARATIVE EVALUATION ===")

for position, sequence in enumerate(
    evaluation_batch_indices,
    start=1,
):

    (
        evaluation_prior_bundle,
        branches,
        rollbacks,
    ) = process_core_batch(
        baseline=evaluation_envelopes[sequence],
        baseline_certificate=evaluation_certificates[sequence],
        context=evaluation_contexts[sequence],
        history=evaluation_histories[sequence],
        prior_record_hash=evaluation_record_predecessors[
            sequence
        ],
        prefix_entries=evaluation_prefix,
        prior_bundle=evaluation_prior_bundle,
        anchor=evaluation_anchor,
        consume=collect_reserved_case,
    )

    evaluation_checkpoints[sequence] = deepcopy(
        evaluation_prior_bundle
    )

    evaluation_branch_count += branches

    evaluation_rollback_count += rollbacks

    if position % 10 == 0 or position == EVALUATION_COUNT:

        print(
            f"Processed {position}/{EVALUATION_COUNT} | "
            f"Decisions: {len(evaluation_comparative_results)}"
        )


# ------------------------------------------------------------
# 7. Validate completeness and history isolation
# ------------------------------------------------------------

expected_evaluation_keys = {
    (sequence, scenario, architecture)
    for sequence in evaluation_batch_indices
    for scenario in CORE_SCENARIOS
    for architecture in ARCHITECTURES
}

observed_evaluation_keys = [
    (
        row["sequence"],
        row["scenario"],
        row["architecture"],
    )
    for row in evaluation_comparative_results
]

assert len(observed_evaluation_keys) == len(
    expected_evaluation_keys
)

assert len(observed_evaluation_keys) == len(
    set(observed_evaluation_keys)
)

assert set(observed_evaluation_keys) == (
    expected_evaluation_keys
)

assert set(evaluation_control_cases) == set(
    evaluation_batch_indices
)

assert set(evaluation_checkpoints) == set(
    evaluation_batch_indices
)

assert anchor_state(ethereum_anchor) == (
    original_development_state
)

assert evaluation_prefix[:DEVELOPMENT_COUNT] == log_entries

assert evaluation_anchor.count() == (
    DEVELOPMENT_COUNT + EVALUATION_COUNT
)

assert evaluation_branch_count == (
    EVALUATION_COUNT + evaluation_rollback_count
)

for sequence, case in evaluation_control_cases.items():

    assert case["anchor"].count() == sequence + 1

    assert case["anchor"].commitment_at(
        sequence + 1
    ) is None

    assert case["anchor"].commitment_at(sequence).hex() == (
        evaluation_envelopes[sequence]["commitment"]
    )


# ------------------------------------------------------------
# 8. Calculate the frozen evaluation metrics
# ------------------------------------------------------------

evaluation_metrics = {}

for architecture in ARCHITECTURES:

    rows = [
        row for row in evaluation_comparative_results
        if row["architecture"] == architecture
    ]

    evaluation_metrics[architecture] = (
        calculate_architecture_metrics(rows)
    )

print("\n=== RESERVED EVALUATION METRICS ===")

for architecture, metrics in evaluation_metrics.items():

    print(f"\n--- {architecture} ---")

    for metric, (numerator, denominator) in metrics.items():

        rate = (
            f"{100 * numerator / denominator:.2f}%"
            if denominator else "N/A"
        )

        print(
            f"{metric}: "
            f"{numerator}/{denominator} ({rate})"
        )


# ------------------------------------------------------------
# 9. Final accounting
# ------------------------------------------------------------

evaluation_core_digest = freeze_digest(
    evaluation_comparative_results
)

print("\n=== RESERVED CORE VALIDATION ===")
print("Reserved batches:", EVALUATION_COUNT)
print("Core scenario types:", len(CORE_SCENARIOS))
print("Architectures:", len(ARCHITECTURES))
print("Architecture decisions:",
      len(evaluation_comparative_results))
print("Infrastructure branches:", evaluation_branch_count)
print("Alternative branches restored:", evaluation_rollback_count)
print("Missing observations: 0")
print("Duplicate observations: 0")
print("Original development state preserved: PASS")
print("Preceding commitment history preserved: PASS")
print("Future-state exclusion: PASS")
print("Core evaluation results SHA-256:", evaluation_core_digest)

print("\nVALIDATION: PASS")

=== EXPERIMENTAL FREEZE ===
Development notebook fingerprint: PASS
Evaluation protocol fingerprint: PASS
Evaluation code fingerprint: PASS

=== FROZEN IMPLEMENTATION INITIALIZATION ===
Development cell 129/27: EXECUTED
Development cell 2/27: EXECUTED
Development cell 3/27: EXECUTED
Development cell 64/27: EXECUTED
Development cell 5/27: EXECUTED
Development cell 6/27: EXECUTED
Development cell 7/27: EXECUTED
Development cell 8/27: EXECUTED
Development cell 9/27: EXECUTED
Development cell 10/27: EXECUTED
Development cell 11/27: EXECUTED
Development cell 64/27: EXECUTED
Development cell 13/27: EXECUTED
Development cell 14/27: EXECUTED
Development cell 64/27: EXECUTED
Development cell 16/27: EXECUTED
Development cell 17/27: EXECUTED
Development cell 18/27: EXECUTED
Development cell 19/27: EXECUTED
Development cell 20/27: EXECUTED
Development cell 21/27: EXECUTED
Development cell 22/27: EXECUTED
Development cell 23/27: EXECUTED
Development cell 24/27: EXECUTED
Development cell 25/27: EXECU

In [2]:
# ============================================================
# CELL 29 — Reserved robustness evaluation
# ============================================================
# Reference: EVALUATION_PROTOCOL.md, Sections 2, 3 and 5.
#
# Reuse frozen development functions.
# Preserve preceding commitment histories.
# Keep robustness results separate from core metrics.

assert len(evaluation_comparative_results) == (
    EVALUATION_COUNT * len(CORE_SCENARIOS)
    * len(ARCHITECTURES)
)

reserved_original_state = anchor_state(evaluation_anchor)


# ------------------------------------------------------------
# 1. Initialize result collections
# ------------------------------------------------------------

reserved_source_results = []
reserved_contract_results = []
reserved_chain_results = []
reserved_boundary_results = []
reserved_witness_availability = []
reserved_witness_compromise = []
reserved_infrastructure_results = []


def seed_reserved_anchor():
    """Initialize an independent anchor with the development prefix."""

    anchor = EthereumAnchor()

    for sequence, commitment in enumerate(log_entries):

        receipt = anchor.append(
            sequence,
            commitment,
            anchor.head(),
        )

        assert receipt["status"] == 1

    assert anchor_state(anchor) == original_development_state

    return anchor


reserved_contract_anchor = seed_reserved_anchor()

reserved_chain_original = seed_reserved_anchor()
reserved_chain_alternate = seed_reserved_anchor()

reserved_boundary_anchor = seed_reserved_anchor()

reserved_boundary_prefix = list(log_entries)

reserved_boundary_prior = deepcopy(
    witnessed_checkpoints[-1]
)


# ------------------------------------------------------------
# 2. T09a-b: Conflicting source claims
# ------------------------------------------------------------

def evaluate_reserved_source_conflict(
    sequence,
    canonical_case,
    alternate,
):
    """Repeat the validated source-conflict design."""

    original = canonical_case["envelope"]

    context = canonical_case["policy_context"]

    certificate = canonical_case["certificate"]

    assert original["commitment"] != alternate["commitment"]

    for field in (
        "sequence",
        "source_id",
        "previous_commitment",
    ):
        assert original["evidence"][field] == (
            alternate["evidence"][field]
        )

    for envelope in (original, alternate):

        assert verify_signature(
            envelope["evidence"],
            envelope["signature"],
            envelope["public_key"],
        )

        assert is_authorized(
            envelope["evidence"],
            envelope["public_key"],
        )

        assert evidence_commitment(
            envelope["evidence"]
        ) == envelope["commitment"]

        assert evaluate_evidence_policy(
            envelope["evidence"],
            envelope["public_key"],
            certificate,
            context,
        )["decision"] == ACCEPT

    assert canonical_case["anchor"].commitment_at(
        sequence
    ).hex() == original["commitment"]

    assert canonical_case["anchor"].commitment_at(
        sequence + 1
    ) is None

    rows = collect_conflict_results(
        sequence,
        "canonical",
        canonical_case,
    )

    # Acceptance is a precondition for advancing history.
    assert all(
        row["actual"] == ACCEPT for row in rows
    )

    checkpoint = canonical_case["log_support"]["checkpoint"]

    advanced_support = deepcopy(
        canonical_case["log_support"]
    )

    advanced_support["consistency_proof"] = []

    advanced_case = {
        **canonical_case,
        "history": VerifierHistory(
            next_sequence=sequence + 1,
            previous_commitment=original["commitment"],
        ),
        "log_history": RetainedLogCheckpoint(
            size=checkpoint["size"],
            root=checkpoint["root"],
        ),
        "log_support": advanced_support,
    }

    conditions = {
        "T09a_conflict": {
            **advanced_case,
            "envelope": alternate,
        },
        "T09a_replay_control": advanced_case,
        "T09b_conflict": {
            **canonical_case,
            "envelope": alternate,
        },
    }

    for condition, case in conditions.items():

        rows.extend(
            collect_conflict_results(
                sequence,
                condition,
                case,
            )
        )

    return rows


# ------------------------------------------------------------
# 3. Boundary-condition evaluation
# ------------------------------------------------------------

def evaluate_reserved_boundary_group(
    group,
    prefix,
    prior_bundle,
    anchor,
    context,
):
    """Reuse the frozen boundary and verification functions."""

    branch = construct_infrastructure_branch(
        group[0][1]["committed_envelope"],
        prefix,
        prior_bundle,
        anchor,
    )

    for scenario, injected in group:

        case = assemble_verification_case(
            injected,
            branch,
            context,
        )

        results = verify_all_architectures(case)

        for architecture, result in results.items():

            reserved_boundary_results.append({
                "sequence": context.expected_sequence,
                "scenario": scenario,
                "architecture": architecture,
                "reference": BOUNDARY_REFERENCES[scenario],
                "actual": result["decision"],
                "checks": dict(result["checks"]),
                "violations": list(result["violations"]),
                "unresolved": list(result["unresolved"]),
            })

    return branch["log"]["bundle"]


# ------------------------------------------------------------
# 4. Execute reserved robustness experiments
# ------------------------------------------------------------

print("=== RESERVED ROBUSTNESS EVALUATION ===")

for position, sequence in enumerate(
    evaluation_batch_indices, start=1
):

    baseline = evaluation_envelopes[sequence]

    context = evaluation_contexts[sequence]

    certificate = evaluation_certificates[sequence]

    history = evaluation_histories[sequence]

    prior_hash = evaluation_record_predecessors[sequence]

    canonical_case = evaluation_control_cases[sequence]

    baseline_snapshot = deepcopy(baseline)
    certificate_snapshot = deepcopy(certificate)

    alternate = build_conflicting_envelope(
        baseline, prior_hash
    )

    prefix = evaluation_prefix[:sequence]

    prior_bundle = (
        witnessed_checkpoints[-1]
        if sequence == DEVELOPMENT_COUNT
        else evaluation_checkpoints[sequence - 1]
    )

    # --------------------------------------------------------
    # T09a-b: Source conflict and isolated verifiers
    # --------------------------------------------------------

    reserved_source_results.extend(
        evaluate_reserved_source_conflict(
            sequence,
            canonical_case,
            alternate,
        )
    )

    # --------------------------------------------------------
    # T09c and witness sensitivity
    # --------------------------------------------------------

    first_candidate = build_conflict_log_candidate(
        prefix,
        bytes.fromhex(baseline["commitment"]),
    )

    second_candidate = build_conflict_log_candidate(
        prefix,
        bytes.fromhex(alternate["commitment"]),
    )

    for subset in WITNESS_SUBSETS:

        reserved_witness_availability.append(
            evaluate_witness_availability(
                sequence,
                prefix,
                prior_bundle,
                first_candidate,
                baseline["commitment"],
                subset,
            )
        )

        for quorum in FIRST_QUORUMS:

            reserved_witness_compromise.append(
                evaluate_witness_compromise(
                    sequence,
                    prefix,
                    prior_bundle,
                    first_candidate,
                    second_candidate,
                    baseline["commitment"],
                    alternate["commitment"],
                    quorum,
                    subset,
                )
            )

    # --------------------------------------------------------
    # T09d: Contract-level conflict prevention
    # --------------------------------------------------------

    candidates = {
        "original": baseline,
        "alternate": alternate,
    }

    assert reserved_contract_anchor.count() == sequence

    for order in T09_SUBMISSION_ORDERS:

        with temporary_ethereum_branch(
            reserved_contract_anchor
        ):

            reserved_contract_results.append(
                evaluate_contract_conflict(
                    anchor=reserved_contract_anchor,
                    first_envelope=candidates[order[0]],
                    second_envelope=candidates[order[1]],
                    sequence=sequence,
                    submission_order=order,
                )
            )

    append_chain_candidate(
        reserved_contract_anchor,
        baseline,
    )

    # --------------------------------------------------------
    # T09e: Conflicting blockchain histories
    # --------------------------------------------------------

    assert anchor_state(reserved_chain_original) == (
        anchor_state(reserved_chain_alternate)
    )

    original_branch = append_chain_candidate(
        reserved_chain_original,
        baseline,
    )

    with temporary_ethereum_branch(
        reserved_chain_alternate
    ):

        alternate_branch = append_chain_candidate(
            reserved_chain_alternate,
            alternate,
        )

        reserved_chain_results.extend(
            evaluate_chain_pair(
                original=baseline,
                alternate=alternate,
                original_branch=original_branch,
                alternate_branch=alternate_branch,
                history=history,
                context=context,
                certificate=certificate,
            )
        )

    append_chain_candidate(
        reserved_chain_alternate,
        baseline,
    )

    assert anchor_state(reserved_chain_original) == (
        anchor_state(reserved_chain_alternate)
    )

    # --------------------------------------------------------
    # Boundary conditions
    # --------------------------------------------------------

    groups = {}

    assert reserved_boundary_prefix == prefix

    for scenario in BOUNDARY_REFERENCES:

        injected = inject_boundary(
            scenario,
            baseline,
            certificate,
            context,
            history,
            prior_hash,
        )

        committed = injected["committed_envelope"]

        assert verify_signature(
            committed["evidence"],
            committed["signature"],
            committed["public_key"],
        )

        assert evidence_commitment(
            committed["evidence"]
        ) == committed["commitment"]

        groups.setdefault(
            committed["commitment"], []
        ).append((scenario, injected))

    baseline_commitment = baseline["commitment"]

    assert baseline_commitment in groups

    for commitment, group in groups.items():

        if commitment == baseline_commitment:
            continue

        with temporary_ethereum_branch(
            reserved_boundary_anchor
        ):

            evaluate_reserved_boundary_group(
                group,
                reserved_boundary_prefix,
                reserved_boundary_prior,
                reserved_boundary_anchor,
                context,
            )

    reserved_boundary_prior = (
        evaluate_reserved_boundary_group(
            groups[baseline_commitment],
            reserved_boundary_prefix,
            reserved_boundary_prior,
            reserved_boundary_anchor,
            context,
        )
    )

    reserved_boundary_prefix.append(
        bytes.fromhex(baseline_commitment)
    )

    assert reserved_boundary_anchor.count() == sequence + 1

    # --------------------------------------------------------
    # Infrastructure sensitivity
    # --------------------------------------------------------
    # Obtain the preceding genuine transaction from the
    # actual reserved ledger, not from another backend.

    restricted_ledger = canonical_case["anchor"].tester

    current_receipt = restricted_ledger.get_transaction_receipt(
        canonical_case["ethereum_support"]["transaction_hash"]
    )

    preceding_block = restricted_ledger.get_block_by_number(
        current_receipt["block_number"] - 1
    )

    assert len(preceding_block["transactions"]) == 1

    earlier_transaction = preceding_block["transactions"][0]

    restricted_ledger.get_transaction_by_hash(
        earlier_transaction
    )

    for scenario in INFRASTRUCTURE_SCENARIOS:

        case = inject_infrastructure_failure(
            scenario,
            canonical_case,
            earlier_transaction,
        )

        for field in (
            "envelope",
            "history",
            "certificate",
            "policy_context",
        ):
            assert case[field] == canonical_case[field]

        results = verify_all_architectures(case)

        for architecture, result in results.items():

            reference, condition, expected_status = (
                infrastructure_reference(
                    scenario,
                    architecture,
                    sequence,
                )
            )

            reserved_infrastructure_results.append({
                "sequence": sequence,
                "scenario": scenario,
                "architecture": architecture,
                "reference": reference,
                "actual": result["decision"],
                "expected_condition": condition,
                "expected_status": expected_status,
                "observed_status": (
                    result["checks"][condition]
                    if condition is not None else None
                ),
                "checks": dict(result["checks"]),
                "layer_decisions": dict(
                    result["layer_decisions"]
                ),
                "violations": list(result["violations"]),
                "unresolved": list(result["unresolved"]),
            })

    assert baseline == baseline_snapshot
    assert certificate == certificate_snapshot

    if position % 10 == 0 or position == EVALUATION_COUNT:

        print(
            f"Processed {position}/{EVALUATION_COUNT} batches"
        )


# ------------------------------------------------------------
# 5. Derive T09c from the exhaustive compromise experiment
# ------------------------------------------------------------

reserved_quorum_results = [
    row for row in reserved_witness_compromise
    if len(row["compromised"]) <= MAX_COMPROMISED_WITNESSES
]


# ------------------------------------------------------------
# 6. Validate exact experimental configuration coverage
# ------------------------------------------------------------

def validate_reserved_coverage(
    rows,
    fields,
    expected_keys,
):
    """Detect missing or duplicate experimental observations."""

    observed = [
        tuple(row[field] for field in fields)
        for row in rows
    ]

    assert len(observed) == len(set(observed))

    assert set(observed) == expected_keys

    return len(observed)


coverage_specifications = (

    (
        "T09a-b",
        reserved_source_results,
        ("sequence", "condition", "architecture"),
        {
            (s, c, a)
            for s in evaluation_batch_indices
            for c in T09_SOURCE_REFERENCES
            for a in ARCHITECTURES
        },
    ),

    (
        "T09c",
        reserved_quorum_results,
        ("sequence", "first_quorum", "compromised"),
        {
            (s, q, w)
            for s in evaluation_batch_indices
            for q in FIRST_QUORUMS
            for w in WITNESS_SUBSETS
            if len(w) <= MAX_COMPROMISED_WITNESSES
        },
    ),

    (
        "T09d",
        reserved_contract_results,
        ("sequence", "submission_order"),
        {
            (s, order)
            for s in evaluation_batch_indices
            for order in T09_SUBMISSION_ORDERS
        },
    ),

    (
        "T09e",
        reserved_chain_results,
        ("sequence", "condition", "architecture"),
        {
            (s, c, a)
            for s in evaluation_batch_indices
            for c in T09_CHAIN_REFERENCES
            for a in T09_CHAIN_ARCHITECTURES
        },
    ),

    (
        "Boundaries",
        reserved_boundary_results,
        ("sequence", "scenario", "architecture"),
        {
            (s, c, a)
            for s in evaluation_batch_indices
            for c in BOUNDARY_REFERENCES
            for a in ARCHITECTURES
        },
    ),

    (
        "Witness availability",
        reserved_witness_availability,
        ("sequence", "available"),
        {
            (s, w)
            for s in evaluation_batch_indices
            for w in WITNESS_SUBSETS
        },
    ),

    (
        "Witness compromise",
        reserved_witness_compromise,
        ("sequence", "first_quorum", "compromised"),
        {
            (s, q, w)
            for s in evaluation_batch_indices
            for q in FIRST_QUORUMS
            for w in WITNESS_SUBSETS
        },
    ),

    (
        "Infrastructure sensitivity",
        reserved_infrastructure_results,
        ("sequence", "scenario", "architecture"),
        {
            (s, c, a)
            for s in evaluation_batch_indices
            for c in INFRASTRUCTURE_SCENARIOS
            for a in ARCHITECTURES
        },
    ),
)

print("\n=== ROBUSTNESS EXPERIMENT ACCOUNTING ===")

for name, rows, fields, expected in coverage_specifications:

    count = validate_reserved_coverage(
        rows, fields, expected
    )

    print(f"{name}: {count}/{len(expected)} PASS")


# ------------------------------------------------------------
# 7. Verify preservation of canonical histories
# ------------------------------------------------------------

for anchor in (
    evaluation_anchor,
    reserved_contract_anchor,
    reserved_chain_original,
    reserved_chain_alternate,
    reserved_boundary_anchor,
):

    assert anchor_state(anchor) == reserved_original_state

assert anchor_state(ethereum_anchor) == (
    original_development_state
)

assert reserved_boundary_prefix == evaluation_prefix

assert reserved_boundary_prior["checkpoint"] == (
    evaluation_checkpoints[
        evaluation_batch_indices[-1]
    ]["checkpoint"]
)


# ------------------------------------------------------------
# 8. Report architecture decision matrices
# ------------------------------------------------------------

def report_reserved_matrix(
    title,
    rows,
    condition_field,
    conditions,
    architectures,
):
    """Report measured decisions without forcing agreement."""

    print(f"\n=== {title} ===")

    print(
        f"{'Condition':<27}"
        + "".join(
            f"{a:<13}" for a in architectures
        )
    )

    for condition in conditions:

        cells = []

        for architecture in architectures:

            selected = [
                row for row in rows
                if row[condition_field] == condition
                and row["architecture"] == architecture
            ]

            assert len(selected) == EVALUATION_COUNT

            cells.append(summarize_decisions(selected))

        print(
            f"{condition:<27}"
            + "".join(f"{c:<13}" for c in cells)
        )


report_reserved_matrix(
    "T09a-b SOURCE CONFLICTS",
    reserved_source_results,
    "condition",
    T09_SOURCE_REFERENCES,
    ARCHITECTURES,
)

report_reserved_matrix(
    "T09e CONFLICTING BLOCKCHAIN HISTORIES",
    reserved_chain_results,
    "condition",
    T09_CHAIN_REFERENCES,
    T09_CHAIN_ARCHITECTURES,
)

report_reserved_matrix(
    "BOUNDARY CONDITIONS",
    reserved_boundary_results,
    "scenario",
    BOUNDARY_REFERENCES,
    ARCHITECTURES,
)

report_reserved_matrix(
    "INFRASTRUCTURE SENSITIVITY",
    reserved_infrastructure_results,
    "scenario",
    INFRASTRUCTURE_SCENARIOS,
    ARCHITECTURES,
)


# ------------------------------------------------------------
# 9. Report witness and contract sensitivity
# ------------------------------------------------------------

print("\n=== WITNESS AVAILABILITY ===")

for count in range(WITNESS_COUNT + 1):

    rows = [
        row for row in reserved_witness_availability
        if len(row["available"]) == count
    ]

    print(
        f"Available={count}:",
        dict(Counter(row["actual"] for row in rows)),
    )


print("\n=== WITNESS COMPROMISE ===")

for count in range(WITNESS_COUNT + 1):

    rows = [
        row for row in reserved_witness_compromise
        if len(row["compromised"]) == count
    ]

    conflicts = sum(row["both_accepted"] for row in rows)

    print(
        f"Compromised={count}: "
        f"{conflicts}/{len(rows)} conflicting acceptances"
    )


print("\n=== CONTRACT CONFLICT PREVENTION ===")

for order in T09_SUBMISSION_ORDERS:

    rows = [
        row for row in reserved_contract_results
        if row["submission_order"] == order
    ]

    print(
        f"{order}: "
        f"first accepted={sum(r['first_status'] == 1 for r in rows)}, "
        f"conflict accepted={sum(r['conflict_status'] == 1 for r in rows)}, "
        f"replay accepted={sum(r['replay_status'] == 1 for r in rows)}"
    )


# ------------------------------------------------------------
# 10. Preserve and report theoretical disagreements
# ------------------------------------------------------------

reserved_robustness_diagnostics = {

    "Primary quorum safety violations": sum(
        row["both_accepted"]
        for row in reserved_quorum_results
    ),

    "Honest double-signing events": sum(
        len(row["honest_double_signers"])
        for row in reserved_witness_compromise
    ),

    "Witness vote disagreements": sum(
        row["first_valid_votes"] != WITNESS_QUORUM
        or row["second_valid_votes"] != row["expected_second_votes"]
        for row in reserved_witness_compromise
    ),

    "Witness compromise decision disagreements": sum(
        row["first_decision"] != ACCEPT
        or row["second_decision"] != (
            ACCEPT
            if row["expected_second_votes"] >= WITNESS_QUORUM
            else INDETERMINATE
        )
        for row in reserved_witness_compromise
    ),

    "Witness availability disagreements": sum(
        row["actual"] != row["reference"]
        or row["valid_votes"] != len(row["available"])
        for row in reserved_witness_availability
    ),

    "Contract reference disagreements": sum(
        any(
            row[field] != expected
            for field, expected in T09_CONTRACT_REFERENCE.items()
        )
        for row in reserved_contract_results
    ),

    "Contract state violations": sum(
        not (
            row["conflict_state_preserved"]
            and row["replay_state_preserved"]
            and row["first_commitment_preserved"]
            and row["stored_count"] == row["sequence"] + 1
        )
        for row in reserved_contract_results
    ),

    "Unexpected failed-transaction events": sum(
        row["conflict_logs"] != 0
        or row["replay_logs"] != 0
        for row in reserved_contract_results
    ),

    "Contract verification disagreements": sum(
        not (
            row["first_decision"] == ACCEPT
            and row["conflict_decision"] == REJECT
            and row["replay_decision"] == REJECT
        )
        for row in reserved_contract_results
    ),

    "Blockchain history reference disagreements": sum(
        row["actual"] != row["reference"]
        for row in reserved_chain_results
    ),

    "Infrastructure decision disagreements": sum(
        row["actual"] != row["reference"]
        for row in reserved_infrastructure_results
    ),

    "Infrastructure condition disagreements": sum(
        row["observed_status"] != row["expected_status"]
        for row in reserved_infrastructure_results
    ),

    "Infrastructure common-layer disagreements": sum(
        row["layer_decisions"]["common"] != ACCEPT
        for row in reserved_infrastructure_results
    ),

    "Infrastructure policy-layer disagreements": sum(
        row["layer_decisions"].get("policy", ACCEPT) != ACCEPT
        for row in reserved_infrastructure_results
    ),
}

print("\n=== ROBUSTNESS DIAGNOSTICS ===")

for name, count in reserved_robustness_diagnostics.items():
    print(f"{name}: {count}")


# ------------------------------------------------------------
# 11. Fingerprint reserved robustness results
# ------------------------------------------------------------
# T09c is contained within witness compromise results.
# Preserve the complete compromise dataset only once.

reserved_robustness_results = {
    "source_conflicts": reserved_source_results,
    "contract_conflicts": reserved_contract_results,
    "blockchain_histories": reserved_chain_results,
    "boundary_conditions": reserved_boundary_results,
    "witness_availability": reserved_witness_availability,
    "witness_compromise": reserved_witness_compromise,
    "infrastructure_sensitivity": reserved_infrastructure_results,
}

reserved_robustness_digest = freeze_digest(
    reserved_robustness_results
)

print("\n=== RESERVED ROBUSTNESS VALIDATION ===")
print("Reserved batches:", EVALUATION_COUNT)
print("Experiment coverage: PASS")
print("Missing observations: 0")
print("Duplicate observations: 0")
print("Canonical history preservation: PASS")
print("Original development state preservation: PASS")
print("T09c duplicate execution: 0")
print("Results SHA-256:", reserved_robustness_digest)

print(
    "Theoretical expectations:",
    "PASS"
    if not any(reserved_robustness_diagnostics.values())
    else "DISAGREEMENTS RECORDED",
)

print("\nEXECUTION VALIDATION: PASS")

=== RESERVED ROBUSTNESS EVALUATION ===
Processed 10/65 batches
Processed 20/65 batches
Processed 30/65 batches
Processed 40/65 batches
Processed 50/65 batches
Processed 60/65 batches
Processed 65/65 batches

=== ROBUSTNESS EXPERIMENT ACCOUNTING ===
T09a-b: 1560/1560 PASS
T09c: 1300/1300 PASS
T09d: 130/130 PASS
T09e: 520/520 PASS
Boundaries: 2340/2340 PASS
Witness availability: 1040/1040 PASS
Witness compromise: 4160/4160 PASS
Infrastructure sensitivity: 4290/4290 PASS

=== T09a-b SOURCE CONFLICTS ===
Condition                  A1           A2           B1           B2           C1           C2           
canonical                  A            A            A            A            A            A            
T09a_conflict              R            R            R            R            R            R            
T09a_replay_control        R            R            R            R            R            R            
T09b_conflict              A            A            R            R   

In [3]:
# ============================================================
# CELL 30 — Verification performance and storage overhead
# ============================================================
# Reference: EVALUATION_PROTOCOL.md, Section 4.
#
# Measure verification latency using reserved control cases.
# Separate evidence, verification support, retained history,
# trust configuration and persistent infrastructure state.
#
# Local execution only. No public-network performance claims.

from time import perf_counter_ns
from statistics import median, quantiles
from dataclasses import asdict, is_dataclass
import platform


# ------------------------------------------------------------
# 1. Frozen measurement configuration
# ------------------------------------------------------------

TIMING_WARMUPS = 1
TIMING_REPETITIONS = 5

architecture_order = tuple(ARCHITECTURES)

performance_results = []
storage_results = []
performance_disagreements = []

performance_environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "warmups": TIMING_WARMUPS,
    "repetitions": TIMING_REPETITIONS,
    "timing_unit": "nanoseconds",
}


# ------------------------------------------------------------
# 2. Deterministic storage representation
# ------------------------------------------------------------

def storage_representation(value):
    """Convert experimental objects to deterministic JSON."""

    if isinstance(value, bytes):
        return "0x" + value.hex()

    if isinstance(value, datetime):
        return value.isoformat()

    if is_dataclass(value):
        return storage_representation(asdict(value))

    if isinstance(value, dict):
        return {
            str(key): storage_representation(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            storage_representation(item)
            for item in value
        ]

    if value is None or isinstance(
        value, (str, int, bool)
    ):
        return value

    raise TypeError(
        f"Unsupported storage type: {type(value)}"
    )


def storage_bytes(value):
    """Measure UTF-8 bytes in the declared representation."""

    encoded = json.dumps(
        storage_representation(value),
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return len(encoded)


# ------------------------------------------------------------
# 3. Measure architecture-specific storage
# ------------------------------------------------------------

def measure_architecture_storage(
    sequence,
    architecture,
    case,
):
    """Separate common and architecture-specific material."""

    family, uses_policy = ARCHITECTURES[architecture]

    envelope = case["envelope"]

    supporting_material = {}

    trust_configuration = {
        "authorized_sources": authorized_sources
    }

    retained_history = {
        "common": case["history"]
    }

    infrastructure_state = None

    ethereum_slot_value_bytes = None

    if uses_policy:

        supporting_material.update({
            "certificate": case["certificate"],
            "policy_context": case["policy_context"],
        })

        trust_configuration["trusted_issuers"] = (
            trusted_issuers
        )

    if family == "transparency_log":
        supporting_material["log_support"] = (
            case["log_support"]
        )

        retained_history["log_checkpoint"] = (
            case["log_history"]
        )

        trust_configuration.update({
            "operator_public_key": operator_public_key,
            "witness_public_keys": witness_public_keys,
            "witness_quorum": WITNESS_QUORUM,
            "log_id": LOG_ID,
        })

        bundle = evaluation_checkpoints[sequence]

        # Logical retained-state model:
        # ordered commitment leaves, latest signed checkpoint,
        # and the latest checkpoint retained by each witness.
        infrastructure_state = {
            "entries": evaluation_prefix[:sequence + 1],
            "latest_signed_checkpoint": bundle,
            "witness_states": {
                name: bundle["checkpoint"]
                for name in WITNESS_NAMES
            },
        }

    elif family == "ethereum":

        supporting_material["ethereum_support"] = (
            case["ethereum_support"]
        )

        view = case["anchor"]

        assert view.count() == sequence + 1

        trust_configuration.update({
            "contract_address": view.address,
            "relay_address": view.relay,
        })

        infrastructure_state = {
            "count": view.count(),
            "head": view.head(),
            "commitments": {
                str(index): commitment
                for index, commitment in enumerate(
                    evaluation_prefix[:sequence + 1]
                )
            },
        }

        # Logical Solidity storage-slot values:
        # one uint256 count, one bytes32 head,
        # and one bytes32 commitment per occupied position.
        #
        # The immutable relay is not counted as a storage slot.
        ethereum_slot_value_bytes = (
            32 * (2 + view.count())
        )

    elif family != "signatures":

        raise ValueError(
            f"Unexpected architecture family: {family}"
        )

    return {
        "sequence": sequence,
        "architecture": architecture,
        "record_count": len(
            envelope["evidence"]["records"]
        ),

        "canonical_evidence_bytes": len(
            canonical_bytes(envelope["evidence"])
        ),

        "signed_envelope_bytes": storage_bytes(envelope),

        "supporting_material_bytes": (
            storage_bytes(supporting_material)
            if supporting_material else 0
        ),

        "retained_history_bytes": storage_bytes(
            retained_history
        ),

        "trust_configuration_bytes": storage_bytes(
            trust_configuration
        ),

        "persistent_infrastructure_bytes": (
            storage_bytes(infrastructure_state)
            if infrastructure_state is not None else 0
        ),

        "ethereum_slot_value_bytes": (
            ethereum_slot_value_bytes
        ),
    }


# ------------------------------------------------------------
# 4. Retrieve previously observed control decisions
# ------------------------------------------------------------

control_decisions = {
    (row["sequence"], row["architecture"]): row["actual"]
    for row in evaluation_comparative_results
    if row["scenario"] == "T00"
}

assert len(control_decisions) == (
    EVALUATION_COUNT * len(ARCHITECTURES)
)

performance_state_before = anchor_state(evaluation_anchor)

print("=== PERFORMANCE AND STORAGE EVALUATION ===")

for key, value in performance_environment.items():
    print(f"{key}: {value}")


# ------------------------------------------------------------
# 5. Execute warm-ups and timed verification
# ------------------------------------------------------------

for position, sequence in enumerate(
    evaluation_batch_indices
):

    case = evaluation_control_cases[sequence]

    # Storage measurement remains outside the timing loop.
    for architecture in architecture_order:

        storage_results.append(
            measure_architecture_storage(
                sequence,
                architecture,
                case,
            )
        )

    for repetition in range(
        TIMING_WARMUPS + TIMING_REPETITIONS
    ):

        # Rotate order across batches and repetitions.
        offset = (
            position + repetition
        ) % len(architecture_order)

        rotated_order = (
            architecture_order[offset:]
            + architecture_order[:offset]
        )

        for order_position, architecture in enumerate(
            rotated_order
        ):

            start = perf_counter_ns()

            result = verify_architecture(
                architecture,
                **case,
            )

            elapsed = perf_counter_ns() - start

            expected = control_decisions[
                sequence, architecture
            ]

            # Preserve decision discrepancies.
            if result["decision"] != expected:

                performance_disagreements.append({
                    "sequence": sequence,
                    "architecture": architecture,
                    "repetition": repetition,
                    "expected": expected,
                    "actual": result["decision"],
                })

            # Exclude warm-up observations.
            if repetition < TIMING_WARMUPS:
                continue

            assert elapsed > 0

            performance_results.append({
                "sequence": sequence,
                "architecture": architecture,
                "repetition": (
                    repetition - TIMING_WARMUPS + 1
                ),
                "execution_order": order_position,
                "record_count": len(
                    case["envelope"]["evidence"]["records"]
                ),
                "elapsed_ns": elapsed,
                "decision": result["decision"],
            })

    if (position + 1) % 10 == 0 or (
        position + 1 == EVALUATION_COUNT
    ):

        print(
            f"Processed {position + 1}/"
            f"{EVALUATION_COUNT} batches | "
            f"Timed verifications: {len(performance_results)}"
        )


# ------------------------------------------------------------
# 6. Verify experimental completeness
# ------------------------------------------------------------

expected_timing_keys = {
    (sequence, architecture, repetition)
    for sequence in evaluation_batch_indices
    for architecture in ARCHITECTURES
    for repetition in range(1, TIMING_REPETITIONS + 1)
}

validate_reserved_coverage(
    performance_results,
    ("sequence", "architecture", "repetition"),
    expected_timing_keys,
)

expected_storage_keys = {
    (sequence, architecture)
    for sequence in evaluation_batch_indices
    for architecture in ARCHITECTURES
}

validate_reserved_coverage(
    storage_results,
    ("sequence", "architecture"),
    expected_storage_keys,
)

assert anchor_state(evaluation_anchor) == (
    performance_state_before
)

assert anchor_state(ethereum_anchor) == (
    original_development_state
)


# ------------------------------------------------------------
# 7. Summarize latency at batch level
# ------------------------------------------------------------

batch_latency_results = []

for sequence in evaluation_batch_indices:

    for architecture in ARCHITECTURES:

        rows = [
            row for row in performance_results
            if row["sequence"] == sequence
            and row["architecture"] == architecture
        ]

        assert len(rows) == TIMING_REPETITIONS

        batch_latency_results.append({
            "sequence": sequence,
            "architecture": architecture,
            "record_count": rows[0]["record_count"],
            "median_latency_ms": median(
                row["elapsed_ns"] for row in rows
            ) / 1_000_000,
        })


def describe_distribution(values):
    """Return median and inclusive quartiles."""

    values = list(values)

    q1, _, q3 = quantiles(
        values,
        n=4,
        method="inclusive",
    )

    return {
        "n": len(values),
        "median": median(values),
        "q1": q1,
        "q3": q3,
        "iqr": q3 - q1,
    }


latency_summary = {}

print("\n=== VERIFICATION LATENCY ===")
print("Unit: milliseconds")
print("Analysis unit: median of five runs per batch")

for architecture in ARCHITECTURES:

    statistics = describe_distribution(
        row["median_latency_ms"]
        for row in batch_latency_results
        if row["architecture"] == architecture
    )

    latency_summary[architecture] = statistics

    print(
        f"{architecture}: "
        f"median={statistics['median']:.4f} | "
        f"Q1={statistics['q1']:.4f} | "
        f"Q3={statistics['q3']:.4f} | "
        f"IQR={statistics['iqr']:.4f}"
    )


# ------------------------------------------------------------
# 8. Summarize storage requirements
# ------------------------------------------------------------

STORAGE_METRICS = (
    "canonical_evidence_bytes",
    "signed_envelope_bytes",
    "supporting_material_bytes",
    "retained_history_bytes",
    "trust_configuration_bytes",
    "persistent_infrastructure_bytes",
    "ethereum_slot_value_bytes",
)

storage_summary = {}

print("\n=== STORAGE REQUIREMENTS ===")
print("Unit: bytes")

for architecture in ARCHITECTURES:

    rows = [
        row for row in storage_results
        if row["architecture"] == architecture
    ]

    storage_summary[architecture] = {}

    print(f"\n--- {architecture} ---")

    for metric in STORAGE_METRICS:

        values = [
            row[metric] for row in rows
            if row[metric] is not None
        ]

        if not values:
            continue

        statistics = describe_distribution(values)

        storage_summary[architecture][metric] = statistics

        print(
            f"{metric}: "
            f"median={statistics['median']:.1f} | "
            f"IQR={statistics['iqr']:.1f}"
        )


# ------------------------------------------------------------
# 9. Final validation and result fingerprint
# ------------------------------------------------------------

performance_digest = freeze_digest({
    "environment": performance_environment,
    "timing_results": performance_results,
    "storage_results": storage_results,
    "decision_disagreements": performance_disagreements,
})

print("\n=== PERFORMANCE VALIDATION ===")
print("Reserved control batches:", EVALUATION_COUNT)
print("Architectures:", len(ARCHITECTURES))

print("Warm-up verifications:",
      EVALUATION_COUNT * len(ARCHITECTURES) * TIMING_WARMUPS)

print("Timed verifications:", len(performance_results))
print("Batch-level latency observations:", len(batch_latency_results))
print("Storage observations:", len(storage_results))
print("Decision disagreements:", len(performance_disagreements))
print("Missing observations: 0")
print("Duplicate observations: 0")
print("Canonical history preservation: PASS")
print("Original development state preservation: PASS")
print("Performance results SHA-256:", performance_digest)

print("\nEXECUTION VALIDATION: PASS")

=== PERFORMANCE AND STORAGE EVALUATION ===
python: 3.12.2
platform: macOS-26.2-arm64-arm-64bit
machine: arm64
processor: arm
warmups: 1
repetitions: 5
timing_unit: nanoseconds
Processed 10/65 batches | Timed verifications: 300
Processed 20/65 batches | Timed verifications: 600
Processed 30/65 batches | Timed verifications: 900
Processed 40/65 batches | Timed verifications: 1200
Processed 50/65 batches | Timed verifications: 1500
Processed 60/65 batches | Timed verifications: 1800
Processed 65/65 batches | Timed verifications: 1950

=== VERIFICATION LATENCY ===
Unit: milliseconds
Analysis unit: median of five runs per batch
A1: median=1.2932 | Q1=1.2767 | Q3=1.3599 | IQR=0.0833
A2: median=2.5419 | Q1=2.5048 | Q3=2.6581 | IQR=0.1533
B1: median=6.6242 | Q1=6.5678 | Q3=6.7123 | IQR=0.1445
B2: median=7.8837 | Q1=7.8071 | Q3=7.9999 | IQR=0.1928
C1: median=21.6555 | Q1=4.3499 | Q3=23.9752 | IQR=19.6253
C2: median=23.0257 | Q1=5.6391 | Q3=25.6585 | IQR=20.0194

=== STORAGE REQUIREMENTS ===
Uni

In [4]:
# ============================================================
# CELL 31 — Final analysis and publication figures
# ============================================================
# Reference: EVALUATION_PROTOCOL.md, Sections 5–7.
#
# Paired descriptive comparisons.
# Batch-level sensitivity analysis.
# Publication figures based on measured results.
# Preserve essential research results and fingerprints.

from statistics import correlation
from xml.sax.saxutils import escape


# ------------------------------------------------------------
# 1. Validate input completeness
# ------------------------------------------------------------

assert len(evaluation_comparative_results) == (
    EVALUATION_COUNT * len(CORE_SCENARIOS)
    * len(ARCHITECTURES)
)

assert len(performance_results) == (
    EVALUATION_COUNT * len(ARCHITECTURES)
    * TIMING_REPETITIONS
)

assert len(storage_results) == (
    EVALUATION_COUNT * len(ARCHITECTURES)
)

assert freeze_digest(evaluation_comparative_results) == (
    evaluation_core_digest
)

assert freeze_digest(reserved_robustness_results) == (
    reserved_robustness_digest
)

ARCHITECTURE_NAMES = {
    "A1": "Signatures",
    "A2": "Signatures + rules",
    "B1": "Witnessed log",
    "B2": "Witnessed log + rules",
    "C1": "Ethereum",
    "C2": "Ethereum + rules",
}

PAIRED_COMPARISONS = (
    ("A1", "A2"),
    ("B1", "B2"),
    ("C1", "C2"),
    ("A1", "B1"),
    ("A2", "B2"),
    ("A1", "C1"),
    ("A2", "C2"),
    ("B1", "C1"),
    ("B2", "C2"),
)

record_counts = {
    batch["batch_index"]: len(batch["records"])
    for batch in evaluation_batches
}


# ------------------------------------------------------------
# 2. Paired architecture comparisons
# ------------------------------------------------------------

decision_index = {
    (
        row["sequence"],
        row["scenario"],
        row["architecture"],
    ): row
    for row in evaluation_comparative_results
}

latency_index = {
    (
        row["sequence"],
        row["architecture"],
    ): row["median_latency_ms"]
    for row in batch_latency_results
}

paired_results = []

for first, second in PAIRED_COMPARISONS:

    batch_comparisons = []

    for sequence in evaluation_batch_indices:

        pairs = [
            (
                decision_index[sequence, scenario, first],
                decision_index[sequence, scenario, second],
            )
            for scenario in CORE_SCENARIOS
        ]

        batch_comparisons.append({
            "sequence": sequence,

            "decision_differences": sum(
                left["actual"] != right["actual"]
                for left, right in pairs
            ),

            "agreement_change": sum(
                int(right["agreement"])
                - int(left["agreement"])
                for left, right in pairs
            ),

            "latency_change_ms": (
                latency_index[sequence, second]
                - latency_index[sequence, first]
            ),
        })

    paired_results.append({
        "first": first,
        "second": second,
        "batch_count": len(batch_comparisons),

        "decision_differences": sum(
            row["decision_differences"]
            for row in batch_comparisons
        ),

        "agreement_change": sum(
            row["agreement_change"]
            for row in batch_comparisons
        ),

        "latency_change_ms": describe_distribution(
            row["latency_change_ms"]
            for row in batch_comparisons
        ),

        "batch_results": batch_comparisons,
    })


print("=== PAIRED ARCHITECTURE COMPARISONS ===")

for row in paired_results:

    print(
        f"{row['first']} -> {row['second']} | "
        f"Decision differences={row['decision_differences']} | "
        f"Agreement change={row['agreement_change']:+d} | "
        f"Median latency change="
        f"{row['latency_change_ms']['median']:+.4f} ms"
    )


# ------------------------------------------------------------
# 3. Sensitivity to observed batch size
# ------------------------------------------------------------

def average_ranks(values):
    """Calculate average ranks for tied observations."""

    values = list(values)

    ordered = sorted(
        range(len(values)),
        key=lambda index: values[index],
    )

    ranks = [0.0] * len(values)

    start = 0

    while start < len(ordered):

        end = start + 1

        while (
            end < len(ordered)
            and values[ordered[end]]
            == values[ordered[start]]
        ):
            end += 1

        rank = (start + 1 + end) / 2

        for index in ordered[start:end]:
            ranks[index] = rank

        start = end

    return ranks


def descriptive_spearman(first, second):
    """Return rank correlation without inferential claims."""

    if len(set(first)) < 2 or len(set(second)) < 2:
        return None

    return correlation(
        average_ranks(first),
        average_ranks(second),
    )


batch_size_results = []

for sequence in evaluation_batch_indices:

    for architecture in ARCHITECTURES:

        rows = [
            decision_index[sequence, scenario, architecture]
            for scenario in CORE_SCENARIOS
        ]

        batch_size_results.append({
            "sequence": sequence,
            "architecture": architecture,
            "record_count": record_counts[sequence],

            "agreement_count": sum(
                row["agreement"] for row in rows
            ),

            "scenario_count": len(rows),

            "latency_ms": latency_index[
                sequence, architecture
            ],
        })


batch_size_summary = {}

print("\n=== BATCH-SIZE SENSITIVITY ===")

print("Minimum records:", min(record_counts.values()))
print("Maximum records:", max(record_counts.values()))
print("Median records:", median(record_counts.values()))

for architecture in ARCHITECTURES:

    rows = [
        row for row in batch_size_results
        if row["architecture"] == architecture
    ]

    sizes = [row["record_count"] for row in rows]

    agreement_rho = descriptive_spearman(
        sizes,
        [row["agreement_count"] for row in rows],
    )

    latency_rho = descriptive_spearman(
        sizes,
        [row["latency_ms"] for row in rows],
    )

    batch_size_summary[architecture] = {
        "agreement_spearman": agreement_rho,
        "latency_spearman": latency_rho,
    }

    print(
        f"{architecture}: "
        f"Agreement rho={agreement_rho}, "
        f"Latency rho={latency_rho}"
    )

print("None indicates undefined correlation.")
print("No arbitrary batch-size thresholds applied.")


# ------------------------------------------------------------
# 4. Consolidate robustness outcomes
# ------------------------------------------------------------

witness_availability_summary = []

witness_compromise_summary = []

for count in range(WITNESS_COUNT + 1):

    available_rows = [
        row for row in reserved_witness_availability
        if len(row["available"]) == count
    ]

    compromise_rows = [
        row for row in reserved_witness_compromise
        if len(row["compromised"]) == count
    ]

    witness_availability_summary.append({
        "available": count,
        "accepted": sum(
            row["actual"] == ACCEPT
            for row in available_rows
        ),
        "total": len(available_rows),
    })

    witness_compromise_summary.append({
        "compromised": count,
        "both_accepted": sum(
            row["both_accepted"]
            for row in compromise_rows
        ),
        "total": len(compromise_rows),
    })


source_index = {
    (
        row["sequence"],
        row["condition"],
        row["architecture"],
    ): row["actual"]
    for row in reserved_source_results
}

chain_index = {
    (
        row["sequence"],
        row["condition"],
        row["architecture"],
    ): row["actual"]
    for row in reserved_chain_results
}

conflict_summary = {}

for architecture in ARCHITECTURES:

    shared_history_conflicts = sum(
        source_index[s, "canonical", architecture] == ACCEPT
        and source_index[s, "T09b_conflict", architecture] == ACCEPT
        for s in evaluation_batch_indices
    )

    split_chain_conflicts = None

    if architecture in T09_CHAIN_ARCHITECTURES:

        split_chain_conflicts = sum(
            chain_index[
                s, "original_on_original", architecture
            ] == ACCEPT

            and chain_index[
                s, "alternate_on_alternate", architecture
            ] == ACCEPT

            for s in evaluation_batch_indices
        )

    conflict_summary[architecture] = {
        "shared_history_conflicts": shared_history_conflicts,
        "split_chain_conflicts": split_chain_conflicts,
        "total": EVALUATION_COUNT,
    }


# ------------------------------------------------------------
# 5. Publication design system
# ------------------------------------------------------------
# Minimal vector graphics without additional dependencies.
# Color is supplementary to labels and numerical values.

FIGURE_INK = "#263238"
FIGURE_MUTED = "#627078"
FIGURE_LINE = "#D8DFE2"
FIGURE_LIGHT = "#F1F4F5"

FIGURE_COLORS = {
    "A1": "#77858B",
    "A2": "#455A64",
    "B1": "#80B0AC",
    "B2": "#367D78",
    "C1": "#90A9C5",
    "C2": "#547DAF",
}


class PublicationSVG:
    """Small standard-library vector figure builder."""

    def __init__(self, width, height):

        self.elements = [
            f'<svg xmlns="http://www.w3.org/2000/svg" '
            f'width="{width}" height="{height}" '
            f'viewBox="0 0 {width} {height}">'
        ]

        self.rect(0, 0, width, height, "#FFFFFF")

    def text(
        self, x, y, value, size=13,
        color=FIGURE_INK, weight=400, anchor="start",
    ):

        self.elements.append(
            f'<text x="{x}" y="{y}" '
            f'font-family="Arial, Helvetica, sans-serif" '
            f'font-size="{size}" font-weight="{weight}" '
            f'fill="{color}" text-anchor="{anchor}">'
            f'{escape(str(value))}</text>'
        )

    def rect(
        self, x, y, width, height,
        fill, stroke="none",
    ):

        self.elements.append(
            f'<rect x="{x}" y="{y}" width="{width}" '
            f'height="{height}" fill="{fill}" '
            f'stroke="{stroke}" rx="3"/>'
        )

    def line(
        self, x1, y1, x2, y2,
        color=FIGURE_LINE, width=1,
    ):

        self.elements.append(
            f'<line x1="{x1}" y1="{y1}" '
            f'x2="{x2}" y2="{y2}" '
            f'stroke="{color}" stroke-width="{width}"/>'
        )

    def heading(self, title, subtitle):

        self.text(25, 30, title, 20, weight=600)

        self.text(
            25, 51, subtitle,
            12, FIGURE_MUTED,
        )

    def render(self):

        return "\n".join(
            self.elements + ["</svg>"]
        )


def horizontal_panel(
    figure,
    x,
    y,
    title,
    observations,
    maximum,
    suffix="%",
    bar_width=280,
):
    """Draw labeled horizontal bars."""

    figure.text(x, y, title, 15, weight=600)

    for index, (label, value, color) in enumerate(
        observations
    ):

        row_y = y + 20 + index * 28

        figure.text(
            x, row_y + 14, label, 12
        )

        if value is None:

            figure.text(
                x + 40, row_y + 14,
                "Not tested", 11, FIGURE_MUTED,
            )

            continue

        figure.rect(
            x + 40, row_y,
            bar_width, 18, FIGURE_LIGHT,
        )

        figure.rect(
            x + 40, row_y,
            bar_width * value / maximum,
            18, color,
        )

        figure.text(
            x + bar_width + 49,
            row_y + 14,
            f"{value:.1f}{suffix}",
            12,
        )


figures = {}


# ------------------------------------------------------------
# Figure 1 — Comparative assurance architecture
# ------------------------------------------------------------

fig = PublicationSVG(1000, 410)

fig.heading(
    "Comparative evidence assurance framework",
    "Common signed evidence; different commitment "
    "infrastructure and decision rules",
)

fig.rect(250, 75, 500, 42, FIGURE_LIGHT)

fig.text(
    500, 101,
    "Canonical evidence + source authentication + retained history",
    13, weight=600, anchor="middle",
)

families = (
    ("Digital signatures", "A1", "A2"),
    ("Witnessed transparency log", "B1", "B2"),
    ("Ethereum anchoring", "C1", "C2"),
)

for index, (name, first, second) in enumerate(families):

    x = 25 + index * 325

    center = x + 150

    fig.line(500, 117, 500, 135)
    fig.line(500, 135, center, 135)
    fig.line(center, 135, center, 150)

    fig.rect(
        x, 150, 300, 50,
        "#FFFFFF", FIGURE_COLORS[second],
    )

    fig.text(
        center, 180, name, 14,
        weight=600, anchor="middle",
    )

    for offset, architecture, label in (
        (0, first, "Cryptographic checks"),
        (155, second, "Checks + evidence rules"),
    ):

        fig.rect(
            x + offset, 225, 145, 62,
            FIGURE_COLORS[architecture],
        )

        fig.text(
            x + offset + 72.5,
            248, architecture, 15,
            "#FFFFFF", 600, "middle",
        )

        fig.text(
            x + offset + 72.5,
            269, label, 10,
            "#FFFFFF", 400, "middle",
        )

    fig.line(center, 200, center, 212)
    fig.line(x + 72.5, 212, x + 227.5, 212)
    fig.line(x + 72.5, 212, x + 72.5, 225)
    fig.line(x + 227.5, 212, x + 227.5, 225)

    for center_x in (x + 72.5, x + 227.5):
        fig.line(center_x, 287, center_x, 320)

    fig.line(x + 72.5, 320, x + 227.5, 320)
    fig.line(center, 320, center, 335)

fig.rect(175, 335, 650, 42, FIGURE_LIGHT)

fig.text(
    500, 361,
    "Decision: ACCEPT / REJECT / INDETERMINATE",
    14, weight=600, anchor="middle",
)

figures["figure_01_architectures.svg"] = fig.render()


# ------------------------------------------------------------
# Figure 2 — Chronological experimental design
# ------------------------------------------------------------

fig = PublicationSVG(1000, 300)

fig.heading(
    "Chronological experimental allocation",
    "Original measurement identities preserved; "
    "no overlapping records",
)

for x, count, records, title, color in (
    (
        25, DEVELOPMENT_COUNT, len(development_hashes),
        "Development", "#77858B",
    ),
    (
        500, EVALUATION_COUNT, len(evaluation_hashes),
        "Reserved evaluation", "#367D78",
    ),
):

    fig.rect(x, 85, 475, 65, color)

    fig.text(
        x + 237.5, 110, title,
        16, "#FFFFFF", 600, "middle",
    )

    fig.text(
        x + 237.5, 133,
        f"{count} batches | {records:,} measurements",
        13, "#FFFFFF", 400, "middle",
    )

fig.text(25, 170, "Batch 0", 12)
fig.text(500, 170, "Batch 65", 12, anchor="middle")
fig.text(975, 170, "Batch 129", 12, anchor="end")

fig.line(262.5, 180, 262.5, 205)
fig.line(737.5, 180, 737.5, 205)

fig.rect(25, 205, 460, 45, FIGURE_LIGHT)
fig.rect(515, 205, 460, 45, FIGURE_LIGHT)

fig.text(
    255, 232, "Implementation, validation and freeze",
    13, weight=600, anchor="middle",
)

fig.text(
    745, 232,
    f"{len(evaluation_comparative_results):,} core decisions "
    "+ robustness and performance",
    13, weight=600, anchor="middle",
)

fig.text(
    25, 280,
    "Reserved comparative evaluation; "
    "not a fully blinded holdout.",
    12, FIGURE_MUTED,
)

figures["figure_02_experimental_design.svg"] = fig.render()


# ------------------------------------------------------------
# Figure 3 — Comparative decision reliability
# ------------------------------------------------------------

fig = PublicationSVG(1000, 545)

fig.heading(
    "Comparative decision reliability",
    "Reserved evaluation | architecture-level "
    "rates with predefined denominators",
)

figure_metrics = (
    ("Decision agreement", 25, 90),
    ("False acceptance", 520, 90),
    ("Violation recognition", 25, 325),
    ("Missing evidence recognition", 520, 325),
)

for metric, x, y in figure_metrics:

    observations = []

    for architecture in ARCHITECTURES:

        numerator, denominator = (
            evaluation_metrics[architecture][metric]
        )

        observations.append((
            architecture,
            100 * numerator / denominator,
            FIGURE_COLORS[architecture],
        ))

    horizontal_panel(
        fig, x, y, metric,
        observations, 100,
    )

figures["figure_03_decision_reliability.svg"] = fig.render()


# ------------------------------------------------------------
# Figure 4 — Witness sensitivity
# ------------------------------------------------------------

fig = PublicationSVG(1000, 310)

fig.heading(
    "Witness availability and compromise",
    f"{WITNESS_QUORUM}-of-{WITNESS_COUNT} quorum | "
    "exhaustive witness identity configurations",
)

horizontal_panel(
    fig, 25, 95,
    "Checkpoint acceptance (%)",
    [
        (
            str(row["available"]),
            100 * row["accepted"] / row["total"],
            "#367D78",
        )
        for row in witness_availability_summary
    ],
    100,
)

horizontal_panel(
    fig, 520, 95,
    "Conflicting checkpoint acceptance (%)",
    [
        (
            str(row["compromised"]),
            100 * row["both_accepted"] / row["total"],
            "#B66A57",
        )
        for row in witness_compromise_summary
    ],
    100,
)

fig.text(
    25, 270, "Number of available witnesses",
    12, FIGURE_MUTED,
)

fig.text(
    520, 270, "Number of compromised witnesses",
    12, FIGURE_MUTED,
)

fig.text(
    25, 295,
    "Configuration frequencies, "
    "not real-world attack probabilities.",
    12, FIGURE_MUTED,
)

figures["figure_04_witness_sensitivity.svg"] = fig.render()


# ------------------------------------------------------------
# Figure 5 — Conflicting claim acceptance
# ------------------------------------------------------------

fig = PublicationSVG(1000, 330)

fig.heading(
    "Conflicting evidence across isolated verifiers",
    "Both conflicting claims accepted at "
    "the same sequence position",
)

horizontal_panel(
    fig, 25, 95,
    "Same canonical infrastructure (%)",
    [
        (
            architecture,
            100 * row["shared_history_conflicts"] / row["total"],
            FIGURE_COLORS[architecture],
        )
        for architecture, row in conflict_summary.items()
    ],
    100,
)

horizontal_panel(
    fig, 520, 95,
    "Different local blockchain histories (%)",
    [
        (
            architecture,
            (
                100 * row["split_chain_conflicts"] / row["total"]
                if row["split_chain_conflicts"] is not None
                else None
            ),
            FIGURE_COLORS[architecture],
        )
        for architecture, row in conflict_summary.items()
    ],
    100,
)

fig.text(
    25, 315,
    "Controlled trust conditions; "
    "public-network consensus was not evaluated.",
    12, FIGURE_MUTED,
)

figures["figure_05_conflicting_evidence.svg"] = fig.render()


# ------------------------------------------------------------
# Figure 6 — Verification latency and support size
# ------------------------------------------------------------

fig = PublicationSVG(1000, 330)

fig.heading(
    "Verification cost and supporting evidence",
    "Local verification latency and "
    "deterministic supporting-material representation",
)

latency_maximum = max(
    row["q3"] for row in latency_summary.values()
)

fig.text(
    25, 95, "Verification latency (ms)",
    15, weight=600,
)

for index, architecture in enumerate(ARCHITECTURES):

    statistics = latency_summary[architecture]

    y = 115 + index * 28

    fig.text(25, y + 14, architecture, 12)

    for value, height in (
        (statistics["median"], 18),
    ):

        fig.rect(
            65, y,
            280 * value / latency_maximum,
            height,
            FIGURE_COLORS[architecture],
        )

    q1 = 65 + 280 * statistics["q1"] / latency_maximum
    q3 = 65 + 280 * statistics["q3"] / latency_maximum

    fig.line(q1, y + 9, q3, y + 9, FIGURE_INK, 1.5)
    fig.line(q1, y + 4, q1, y + 14, FIGURE_INK)
    fig.line(q3, y + 4, q3, y + 14, FIGURE_INK)

    fig.text(
        365, y + 14,
        f"{statistics['median']:.3f}", 12,
    )

support_observations = [
    (
        architecture,
        storage_summary[architecture][
            "supporting_material_bytes"
        ]["median"],
        FIGURE_COLORS[architecture],
    )
    for architecture in ARCHITECTURES
]

horizontal_panel(
    fig, 520, 95,
    "Supporting material (bytes)",
    support_observations,
    max(1, max(row[1] for row in support_observations)),
    suffix="",
)

fig.text(
    25, 305,
    "Latency: median and interquartile range.",
    12, FIGURE_MUTED,
)

fig.text(
    520, 305,
    "Support size excludes common signed evidence.",
    12, FIGURE_MUTED,
)

figures["figure_06_verification_cost.svg"] = fig.render()


# ------------------------------------------------------------
# 6. Preserve complete evaluation results
# ------------------------------------------------------------

final_results = {
    "experimental_freeze": evaluation_freeze,

    "development_metrics": development_metrics,

    "evaluation_metrics": evaluation_metrics,

    "core_results": evaluation_comparative_results,

    "robustness_results": reserved_robustness_results,

    "robustness_diagnostics": reserved_robustness_diagnostics,

    "paired_results": paired_results,

    "batch_size_results": batch_size_results,

    "batch_size_summary": batch_size_summary,

    "witness_availability_summary":
        witness_availability_summary,

    "witness_compromise_summary":
        witness_compromise_summary,

    "conflict_summary": conflict_summary,

    "performance_environment": performance_environment,

    "performance_results": performance_results,

    "performance_disagreements": performance_disagreements,

    "batch_latency_results": batch_latency_results,

    "latency_summary": latency_summary,

    "storage_results": storage_results,

    "storage_summary": storage_summary,

    "fingerprints": {
        "development_code": development_code_digest,
        "reference_definitions": reference_definition_digest,
        "core_evaluation": evaluation_core_digest,
        "robustness_evaluation": reserved_robustness_digest,
        "performance": performance_digest,
    },
}

# Serialize and validate before creating output files.

result_text = json.dumps(
    final_results,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
)

assert len(figures) == 6

output_directory = (
    evaluation_root / "results" / "reserved_evaluation"
)

assert not output_directory.exists(), (
    "Evaluation outputs already exist. "
    "Preserve the previous run before repeating."
)

output_directory.mkdir(parents=True)

result_path = output_directory / "evaluation_results.json"

result_path.write_text(
    result_text, encoding="utf-8"
)

for filename, content in figures.items():

    path = output_directory / filename

    path.write_text(
        content, encoding="utf-8"
    )


# ------------------------------------------------------------
# 7. Output integrity manifest
# ------------------------------------------------------------

output_hashes = {
    path.name: file_digest(path)
    for path in sorted(output_directory.iterdir())
    if path.is_file()
}

manifest_path = output_directory / "SHA256SUMS"

manifest_path.write_text(
    "\n".join(
        f"{digest}  {filename}"
        for filename, digest in output_hashes.items()
    ) + "\n",
    encoding="utf-8",
)

assert len(output_hashes) == 7

assert all(
    file_digest(output_directory / filename) == digest
    for filename, digest in output_hashes.items()
)

print("\n=== PUBLICATION OUTPUTS ===")

for filename, digest in output_hashes.items():

    print(filename)
    print("  SHA-256:", digest)

print("\n=== FINAL EVALUATION ACCOUNTING ===")

print("Reserved batches:", EVALUATION_COUNT)

print("Core architecture decisions:",
      len(evaluation_comparative_results))

print("Paired architecture comparisons:",
      len(paired_results))

print("Timed verifications:",
      len(performance_results))

print("Storage observations:",
      len(storage_results))

print("Publication figures:", len(figures))

print("Machine-readable result datasets: 1")

print("Output integrity manifest: PASS")

print("Performance decision disagreements:",
      len(performance_disagreements))

print("Output directory:", output_directory)

print("\nEXECUTION VALIDATION: PASS")

=== PAIRED ARCHITECTURE COMPARISONS ===
A1 -> A2 | Decision differences=585 | Agreement change=+585 | Median latency change=+1.2499 ms
B1 -> B2 | Decision differences=585 | Agreement change=+585 | Median latency change=+1.2545 ms
C1 -> C2 | Decision differences=585 | Agreement change=+585 | Median latency change=+1.3374 ms
A1 -> B1 | Decision differences=0 | Agreement change=+0 | Median latency change=+5.3175 ms
A2 -> B2 | Decision differences=0 | Agreement change=+0 | Median latency change=+5.3296 ms
A1 -> C1 | Decision differences=0 | Agreement change=+0 | Median latency change=+20.3492 ms
A2 -> C2 | Decision differences=0 | Agreement change=+0 | Median latency change=+20.4617 ms
B1 -> C1 | Decision differences=0 | Agreement change=+0 | Median latency change=+15.0323 ms
B2 -> C2 | Decision differences=0 | Agreement change=+0 | Median latency change=+15.1547 ms

=== BATCH-SIZE SENSITIVITY ===
Minimum records: 24
Maximum records: 695
Median records: 36
A1: Agreement rho=None, Latency r